# INFO812 Data Mining — Group Assignment
## Credit Card Fraud Detection: Decision Tree & Logistic Regression Classifier Building
**Course:** INFO812 Data Mining (Trimester 3, 2026)  
**Workflow:** Scikit-Learn 70/30 Train-Test Split with Feature Selection, Class Balancing, Evaluation & Tree Visualisation.

---
### Notebook Table of Contents:
1. **Importing Required Libraries**
2. **Loading Dataset & Statistical Summary (Q1 – Q3)**
3. **Data Visualisations (Q4)**
4. **Feature Selection (Q6 – Q9)**
5. **Splitting Data: 70% Train and 30% Test (Q10)**
6. **Data Preprocessing & Rebalancing via SMOTE (Q5)**
7. **Building Decision Tree & Logistic Regression Models (Q12)**
8. **Evaluating Models on Test Dataset: Precision, Recall, F1, Confusion Matrix (Q11 & Q13)**
9. **Visualizing Decision Trees & Extracting Rules (Q14 & Appendix D)**
10. **Commercial Cost-Benefit Impact Analysis (Q14)**

### 1. Importing Required Libraries

In [ ]:
# Load libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, export_graphviz, plot_tree, export_text # Decision Tree Classifier
from sklearn.linear_model import LogisticRegression # Logistic Regression Classifier
from sklearn.model_selection import train_test_split # Train/Test Split Function
from sklearn.feature_selection import mutual_info_classif # Information Gain
from sklearn import metrics # Metrics module for evaluation
from imblearn.over_sampling import SMOTE # Class Balancing

# Visual style settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, DejaVu Sans'
plt.rcParams['font.size'] = 10
print('Libraries imported successfully!')

### 2. Loading Data (Q1 – Q3)
We load `creditcard.csv` and inspect transaction shapes and the severe class imbalance.

In [ ]:
# Load dataset
data_path = '../creditcard.csv' if os.path.exists('../creditcard.csv') else 'creditcard.csv'
df = pd.read_csv(data_path)

print(f'Total Rows: {df.shape[0]:,}')
print(f'Total Columns: {df.shape[1]}')

n_legit = (df['Class'] == 0).sum()
n_fraud = (df['Class'] == 1).sum()
print(f'Class 0 (Legitimate): {n_legit:,} ({n_legit/len(df)*100:.4f}%)')
print(f'Class 1 (Fraudulent): {n_fraud:,} ({n_fraud/len(df)*100:.4f}%)')
print(f'Imbalance Ratio: {n_legit/n_fraud:.1f} : 1')
print(f'Missing Values: {df.isnull().sum().sum()}')

# First 5 transactions
df.head()

### 3. Data Visualisations (Q4)
We plot the class imbalance bar chart, transaction amount distribution, and key predictive feature densities.

In [ ]:
# Visualisation 1: Class Imbalance (Log Scale) & Amount Boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
palette = ['#1f77b4', '#d62728']

axes[0].bar(['Legitimate (0)', 'Fraudulent (1)'], [n_legit, n_fraud], color=palette, edgecolor='black', width=0.5)
axes[0].set_yscale('log')
axes[0].set_title('Transaction Class Imbalance (Log Scale)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1], palette=palette, showfliers=False, width=0.4)
axes[1].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
axes[1].set_title('Transaction Amount Distribution by Class', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amount (EUR)')
plt.tight_layout()
plt.show()

### 4. Feature Selection (Q6 – Q9)
We identify the most predictive attributes using Information Gain and Correlation ranking.

In [ ]:
# Feature Selection using Top 11 Predictive Attributes
feature_cols = ['V14', 'V17', 'V12', 'V10', 'V11', 'V16', 'V4', 'V3', 'V7', 'V9', 'Amount']
X = df[feature_cols] # Features
y = df['Class']      # Target variable

print('Selected Feature Columns:', feature_cols)
X.head()

### 5. Splitting Data (70% Training and 30% Test)
We partition the dataset into 70% training and 30% unseen testing, using stratification to preserve class balance.

In [ ]:
# Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Training Instances: {X_train.shape[0]:,} (Class counts: {np.bincount(y_train)})')
print(f'Testing Instances:  {X_test.shape[0]:,}  (Class counts: {np.bincount(y_test)})')

### 6. Preprocessing & SMOTE Balancing on Training Set (Q5)
We apply SMOTE **strictly to the training partition** to rebalance minority class examples and prevent data leakage.

In [ ]:
# Apply SMOTE on training data
smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Pre-SMOTE Training Class Counts:  {np.bincount(y_train)}')
print(f'Post-SMOTE Training Class Counts: {np.bincount(y_train_res)}')

### 7. Building Models: Decision Tree & Logistic Regression (Q12)
We train both classifiers on the preprocessed training set.

In [ ]:
# 1. Create Decision Tree classifier object (C4.5 / J48 entropy criterion)
clf_tree = DecisionTreeClassifier(criterion='entropy', max_depth=4, min_samples_leaf=5, random_state=42)
clf_tree = clf_tree.fit(X_train_res, y_train_res)
y_pred_tree = clf_tree.predict(X_test)
y_prob_tree = clf_tree.predict_proba(X_test)[:, 1]

# 2. Create Logistic Regression classifier object (Comparative baseline)
clf_log = LogisticRegression(max_iter=1000, random_state=42)
clf_log = clf_log.fit(X_train_res, y_train_res)
y_pred_log = clf_log.predict(X_test)
y_prob_log = clf_log.predict_proba(X_test)[:, 1]

print('Both models successfully trained!')

### 8. Evaluating the Models on 30% Test Dataset (Q11 & Q13)
We evaluate Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and Classification Reports.

In [ ]:
print('=' * 65)
print('EVALUATION ON 30% UNSEEN TEST DATA (85,443 TRANSACTIONS)')
print('=' * 65)

print('\n--- DECISION TREE CLASSIFIER ---')
print(f'Accuracy:        {metrics.accuracy_score(y_test, y_pred_tree)*100:.2f}%')
print(f'Fraud Precision: {metrics.precision_score(y_test, y_pred_tree)*100:.2f}%')
print(f'Fraud Recall:    {metrics.recall_score(y_test, y_pred_tree)*100:.2f}%')
print(f'Fraud F1-Score:  {metrics.f1_score(y_test, y_pred_tree):.4f}')
print(f'ROC-AUC:         {metrics.roc_auc_score(y_test, y_prob_tree):.4f}')
print('\nClassification Report:\n', metrics.classification_report(y_test, y_pred_tree, target_names=['Legitimate', 'Fraud']))

print('--- LOGISTIC REGRESSION CLASSIFIER ---')
print(f'Accuracy:        {metrics.accuracy_score(y_test, y_pred_log)*100:.2f}%')
print(f'Fraud Precision: {metrics.precision_score(y_test, y_pred_log)*100:.2f}%')
print(f'Fraud Recall:    {metrics.recall_score(y_test, y_pred_log)*100:.2f}%')
print(f'Fraud F1-Score:  {metrics.f1_score(y_test, y_pred_log):.4f}')
print(f'ROC-AUC:         {metrics.roc_auc_score(y_test, y_prob_log):.4f}')
print('\nClassification Report:\n', metrics.classification_report(y_test, y_pred_log, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Side-by-Side Confusion Matrices
cm_t = metrics.confusion_matrix(y_test, y_pred_tree)
cm_l = metrics.confusion_matrix(y_test, y_pred_log)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=150)
sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Pred Legit (0)', 'Pred Fraud (1)'], yticklabels=['Actual Legit (0)', 'Actual Fraud (1)'])
axes[0].set_title('Decision Tree Confusion Matrix', fontweight='bold')

sns.heatmap(cm_l, annot=True, fmt='d', cmap='Reds', cbar=False, ax=axes[1],
            xticklabels=['Pred Legit (0)', 'Pred Fraud (1)'], yticklabels=['Actual Legit (0)', 'Actual Fraud (1)'])
axes[1].set_title('Logistic Regression Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

### 9. Visualizing Decision Trees & Extracting Rules (Q14 & Appendix D)
We visualize the tree graphically and extract human-interpretable IF-THEN rules.

In [ ]:
# Visualizing Decision Tree with Matplotlib plot_tree
plt.figure(figsize=(20, 10), dpi=200)
plot_tree(
    clf_tree,
    feature_names=feature_cols,
    class_names=['Legitimate', 'Fraud'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Credit Card Fraud Detection — Decision Tree Architecture', fontsize=14, fontweight='bold', pad=15)
plt.show()

# Extract Text Rules
rules_text = export_text(clf_tree, feature_names=feature_cols)
print('Sample Extracted IF-THEN Rules (First 20 lines):\n')
print('\n'.join(rules_text.split('\n')[:20]))

### 10. Commercial Financial Cost Impact Analysis (Q14 — 30 Marks)
Applying banking costs: **False Negative (Unrecovered Fraud) = €250**, **False Positive (Customer Friction) = €15**.

In [ ]:
fn_cost, fp_cost = 250, 15

cost_tree = (cm_t[1, 0] * fn_cost) + (cm_t[0, 1] * fp_cost)
cost_log = (cm_l[1, 0] * fn_cost) + (cm_l[0, 1] * fp_cost)

print('=' * 65)
print('COMMERCIAL FINANCIAL COST IMPACT (Q14)')
print('=' * 65)
print(f'Decision Tree Total Loss:        €{cost_tree:,} (FN: €{cm_t[1,0]*fn_cost:,} | FP: €{cm_t[0,1]*fp_cost:,})')
print(f'Logistic Regression Total Loss:  €{cost_log:,} (FN: €{cm_l[1,0]*fn_cost:,} | FP: €{cm_l[0,1]*fp_cost:,})')
print(f'Difference:                      €{abs(cost_log - cost_tree):,}')
print('=' * 65)